# Feature Engineering II: Column Transformers and Pipelines
<img src = "./images/lego.webp" width = "450">

<a href = "https://www.highsnobiety.com/p/lego-transformers-optimus-prime/">Image Source</a>

This notebook build on the [feature engineering introduction notebook](1_intro_to_fe.ipynb) to automate the transformation process, simplifying our workflow and unlocking the potential of the sklearn library.
<hr style="border:2px solid black">

## Penguin Dataset: Episode V - The Flipper Strikes Back

We will use the Palmer Penguin Dataset.

### Business Goal
> Predict the penguin body mass given the input feature : flipper_length_mm, bill_length_mm, species and sex

#### Load Packages

In [24]:
# data analysis stack
import numpy as np
import pandas as pd

# data visualization stack
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('whitegrid')

# machine-learning stack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    KBinsDiscretizer,
    PolynomialFeatures
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# miscellaneous
import warnings
warnings.filterwarnings("ignore")

#### Load Data

In [4]:
df = pd.read_csv('../data/penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


#### Features and Target

In [19]:
numerical_features = [
    'flipper_length_mm',
    'bill_length_mm'
]

categorical_features = [
    'species',
    'sex'
]

features = numerical_features + categorical_features

target_variable = 'body_mass_g'

#### Feature-Target separation

In [20]:
# Feature matrix 
X = df[features]

# Target column
y = df[target_variable]

#### Train-Test Split

In [272]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=88, shuffle=True, stratify=X['species'])

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (256, 4)
Test shape: (86, 4)


## Exploratory Data Analysis
Check issue with data
+ which variable has missing values?
+ which variables are binary, categorical, metric?
+ do categorical variables have non-numeric values?
+ do metric features are varying on a different scale?


In [273]:
# Assuming X_train is a DataFrame and y_train is a Series
df_train = pd.concat([X_train, y_train], axis=1)

print("Combined train data shape:", df_train.shape)

Combined train data shape: (256, 5)


In [274]:
df_train.isna().sum()

flipper_length_mm    1
bill_length_mm       0
species              0
sex                  5
body_mass_g          0
dtype: int64

<hr style="border:2px solid black">

## Feature Engineering
We have a pair of tools, `ColumnTransformer()` and `Pipeline()`, which can dramatically simplify and automate feature engineering.

The above code will config all the output of the transformation to a pandas DataFrame


### `ColumnTransformer()`
<a href="https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html">`ColumnTransformer` </a> allows us to specify which columns receive which transformations (and conveniently reintegrates the dataset).

Parameters:
 * `transformers` - list of tuples `(name, transformer, columns)`

 * `remainder` - used as last tuple if there are any untouched columns. Choose either `drop` or `passthrough`.
  
**Note** that `ColumnTransformer()` runs all transformers in parallel, not sequentially, so if a column is transformed more than once, the version generated by each of these transformations will be included.

#### Building our first transformer


In [275]:
# define our transformers - name, method, target
transformers = [('ohe', OneHotEncoder(drop = 'first',sparse_output=False), ['species', 'sex']),
                ('bill_scaler', RobustScaler(), ['bill_length_mm']),
                ('flip_scaler', RobustScaler(), ['flipper_length_mm'])]

In [276]:
# now we instantiate our ColumnTransformer() object
column_transformer = ColumnTransformer(transformers,
                                       remainder = 'drop')
column_transformer

ColumnTransformer(transformers=[('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['species', 'sex']),
                                ('bill_scaler', RobustScaler(),
                                 ['bill_length_mm']),
                                ('flip_scaler', RobustScaler(),
                                 ['flipper_length_mm'])])

We still need to impute missing values in sex and flipper_length_mm, but if we do so in this transformer we will create an imputed copy of sex and flipper_length_mm and a one-hot encoded versions with missing values.

What we need here is a way to sequentially apply transformations, which leads us nicely into...

### `Pipeline()`

<a href="https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html">`Pipeline()`</a> allows us to sequentially apply multiple transformers on the same column(s).


Parameters:
 * `steps` - list of tuples `(name, transformer)`

#### Build a pipeline and integrate it into our transformer

In [277]:
# Let's define the steps to impute and transform sex
sex_steps = [('imputer', SimpleImputer(strategy = 'most_frequent')),
             ('sex_ohe', OneHotEncoder(drop = 'first',sparse_output=False))
             ]

In [278]:
# Let's instantiate the sex pipeline
sex_pipeline = Pipeline(steps=sex_steps)
sex_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('sex_ohe', OneHotEncoder(drop='first', sparse_output=False))])

In [279]:
# Let's define the steps to impute and transform flipper_length_mm
flipper_steps = [('imputer', SimpleImputer(strategy = 'median')),
             ('flipper_scaler', RobustScaler())
             ]

In [280]:
# Let's instantiate the flipper pipeline
flipper_pipeline = Pipeline(steps=flipper_steps)
flipper_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('flipper_scaler', RobustScaler())])

In [281]:
# Let's build a new transformer to include this pipeline
transformers_2 = [('sex_pipeline', sex_pipeline, ['sex']),
                  ('ohe', OneHotEncoder(drop = 'first',sparse_output=False), ['species']),
                  ('flipper_pipeline', flipper_pipeline, ['flipper_length_mm']),
                ('scaler',RobustScaler(), ['bill_length_mm'])
                 ]

column_transformer_2 = ColumnTransformer(transformers=transformers_2,
                                         remainder = 'drop').set_output(transform='pandas')
column_transformer_2      

ColumnTransformer(transformers=[('sex_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('sex_ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False))]),
                                 ['sex']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['species']),
                                ('flipper_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('flipper_scaler',
                                                  RobustScaler())]),
                                 ['flipper_length_mm']),
                                ('scaler', RobustScaler(), ['bill_length_mm'])])

#### Let's Try It Out

In [282]:
# Fit the column transformer object ONLY using train data
column_transformer_2.fit(X_train)

ColumnTransformer(transformers=[('sex_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('sex_ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False))]),
                                 ['sex']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['species']),
                                ('flipper_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('flipper_scaler',
                                                  RobustScaler())]),
                                 ['flipper_length_mm']),
                                ('scaler', RobustScaler(), ['bill_length_mm'])])

In [283]:
X_train.isna().sum()

flipper_length_mm    1
bill_length_mm       0
species              0
sex                  5
dtype: int64

In [284]:
# Transform the data
X_train_fe = column_transformer_2.transform(X_train)
X_train_fe

,sex_pipeline__sex_Male,ohe__species_Chinstrap,ohe__species_Gentoo,flipper_pipeline__flipper_length_mm,scaler__bill_length_mm
140,1.0,0.0,0.0,-0.430108,-0.440000
45,1.0,0.0,0.0,-0.645161,-0.382857
193,1.0,1.0,0.0,-0.043011,0.737143
286,1.0,0.0,1.0,1.376344,0.577143
37,0.0,0.0,0.0,-0.688172,-0.782857
...,...,...,...,...,...
245,0.0,0.0,1.0,0.817204,0.005714
31,0.0,0.0,0.0,-0.387097,-0.565714
82,1.0,0.0,0.0,-0.172043,-1.068571
56,1.0,0.0,0.0,-0.172043,-0.440000


## Model Building

### Nesting Pipelines

We've already seen how we can use essentially any named function or object as a step in our pipelines and transformers. The last trick we'll look at with pipelines is the ability to nest several layers within one.

In [285]:
# build a pipeline containing our complete transformer and then a linear regression model
model_steps = [('feature_enginnering', column_transformer_2),
               ('linear_regression', LinearRegression())]
linear_model = Pipeline(steps = model_steps)
linear_model

Pipeline(steps=[('feature_enginnering',
                 ColumnTransformer(transformers=[('sex_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('sex_ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  ['sex']),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['species']),
                                                 ('flipper_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('flipper_scaler',
                                                                   RobustScaler())]),
                                                  ['flipper_length_mm']),
                                                 ('scaler', RobustScaler(),
                                                  ['bill_length_mm'])])),
                ('linear_regression', LinearRegression())])

**train model**

In [286]:
linear_model.fit(X_train,y_train)

Pipeline(steps=[('feature_enginnering',
                 ColumnTransformer(transformers=[('sex_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('sex_ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  ['sex']),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['species']),
                                                 ('flipper_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('flipper_scaler',
                                                                   RobustScaler())]),
                                                  ['flipper_length_mm']),
                                                 ('scaler', RobustScaler(),
                                                  ['bill_length_mm'])])),
                ('linear_regression', LinearRegression())])

In [287]:
training_score = linear_model.score(X_train,y_train)
print(f"training r2 score: {round(training_score, 6)}")

training r2 score: 0.862912


### Model Evaluation

**Model Weigths**

In [288]:
column_step = linear_model.steps[0][1]
column_step

ColumnTransformer(transformers=[('sex_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('sex_ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False))]),
                                 ['sex']),
                                ('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['species']),
                                ('flipper_pipeline',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('flipper_scaler',
                                                  RobustScaler())]),
                                 ['flipper_length_mm']),
                                ('scaler', RobustScaler(), ['bill_length_mm'])])

In [289]:
model_step = linear_model.steps[1][1]
model_step

LinearRegression()

In [290]:
coef_model = pd.DataFrame(data=model_step.coef_.reshape(1,-1), columns=column_step.get_feature_names_out(), index=['weigth'])

coef_model['intercept'] = model_step.intercept_
coef_model

,sex_pipeline__sex_Male,ohe__species_Chinstrap,ohe__species_Gentoo,flipper_pipeline__flipper_length_mm,scaler__bill_length_mm,intercept
weigth,421.527891,-360.368873,628.276557,406.147438,261.616625,3766.922464


**Model Prediction**

In [291]:
y_pred_test = linear_model.predict(X_test)
y_pred_test

array([4748.42170602, 3919.74689153, 4280.91244613, 4271.4746251 ,
       3864.18266383, 4480.41129253, 3426.76642453, 4688.1529015 ,
       4078.84551886, 4746.06811232, 4540.06984327, 3401.435034  ,
       3935.16452114, 5293.33757526, 3397.03297349, 4824.27634257,
       3879.30038771, 4087.34451298, 3065.76124063, 3321.48346383,
       3855.21556154, 3555.96861954, 5504.67934229, 3911.41348926,
       3532.18630969, 4130.14722108, 3271.12580966, 3334.55011032,
       5388.37820191, 3324.16563065, 5284.67298928, 4618.11248167,
       5225.34562224, 3320.54202635, 5417.33580732, 4189.47197755,
       3392.46271056, 4151.0765511 , 4044.38143418, 3404.73006518,
       5517.27527004, 4668.16500896, 5214.32744257, 3339.58848141,
       4015.58942063, 3465.3300534 , 3604.44339876, 3345.56828999,
       3284.18984557, 5559.13393008, 3499.16043805, 3438.88902296,
       3345.56828999, 4679.65390737, 3095.66289409, 4119.76274142,
       4342.12268813, 3371.53599111, 4170.12300617, 4014.01167

**Model Performance**

In [304]:
test_score = linear_model.score(X_test,y_test)
print(f"test r2 score: {round(test_score, 6)}")

test r2 score: 0.872736


<hr style="border:2px solid black">